In [13]:
import nibabel as nib        # For nifti files
import numpy as np           # For matrix math
import SimpleITK as sitk     # For N4 correction
import torchio as tio        # For deep learning
from dcm2niix import main as dcm2niix_run
from pathlib import Path
from dotenv import load_dotenv
import os

In [14]:
# 1. Path to your top ADNI folder

load_dotenv()
folder_path = os.getenv('ADNI_FOLDER_PATH')
raw_root = Path(folder_path)

# 2. Path to where you want all converted .nii.gz files saved
output_root = Path("data/nifti_raw")

# Loop through every Participant folder inside cn cohort/ADNI
for subject_dir in raw_root.iterdir():
    if subject_dir.is_dir():
        print(f"Converting subject: {subject_dir.name}")
        
        # Create a matching subject directory in your output folder
        subj_output = output_root / subject_dir.name
        subj_output.mkdir(parents=True, exist_ok=True)
        
        # dcm2niix will automatically crawl down into MPRAGE -> Visits -> Cryptic ID -> DICOMs
        dcm2niix_run([
            "-z", "y",                 # Compress output to .nii.gz
            "-f", "%i_%p_%s",          # Filename style: ParticipantID_Protocol_Series
            "-o", str(subj_output),    # Destination folder for this participant
            str(subject_dir)           # Input directory (this participant's root folder)
        ])

print("All participant conversions complete!")

Converting subject: 009_S_0751


KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import torch
import torchio as tio
import SimpleITK as sitk
import numpy as np

# =============================================================================
# Pipeline Parameters & Placeholder Dimensions
# =============================================================================
NIFTI_INPUT_DIR = "data/nifti_raw"
PROCESSED_OUTPUT_DIR = "data/preprocessed_tensors"

# Placeholder spatial dimensions (Depth, Height, Width)
DEPTH, HEIGHT, WIDTH = 128, 128, 128


def preprocess_single_volume(nifti_path: Path, target_shape=(128, 128, 128)) -> torch.Tensor:
    """
    Executes the 6-step preprocessing pipeline on a single 3D MRI file.
    Returns a 5D PyTorch tensor: (Batch=1, Channel=1, Depth, Height, Width)
    """
    
    # -------------------------------------------------------------------------
    # STEP 1: Motion Correction (Orientation Standardization) & N4 Bias Field
    # -------------------------------------------------------------------------
    # Read SimpleITK image
    sitk_img = sitk.ReadImage(str(nifti_path), sitk.sitkFloat32)
    
    # Standardize orientation to RAS (Right-Anterior-Superior) coordinate frame
    sitk_img = sitk.DICOMOrient(sitk_img, 'RAS')
    
    # N4 Bias Field Correction to remove scanner intensity variations
    n4_corrector = sitk.N4BiasFieldCorrectionImageFilter()
    sitk_img = n4_corrector.Execute(sitk_img)
    
    # Convert SimpleITK image into a TorchIO Subject for pipeline transformation
    array = sitk.GetArrayFromImage(sitk_img)  # Shape: (D, H, W)
    tensor_4d = torch.from_numpy(array).unsqueeze(0)  # Shape: (C=1, D, H, W)
    
    subject = tio.Subject(mri=tio.ScalarImage(tensor=tensor_4d))

    # -------------------------------------------------------------------------
    # STEPS 2 - 6: TorchIO Preprocessing Sequence
    # -------------------------------------------------------------------------
    transform_pipeline = tio.Compose([
        # STEP 2: Skull Stripping (Masking out non-brain tissue & dark background)
        tio.Clamp(lower=0),  # Zero-out negative artifact values
        tio.Mask(masking_method=lambda x: x > x.mean()),  # Foreground brain mask extraction
        
        # STEP 3: Spatial Normalization (Resample to uniform 1mm x 1mm x 1mm voxel spacing)
        tio.Resample(target=(1.0, 1.0, 1.0)),
        
        # STEP 4: Intensity Normalization (Z-score scaling on foreground brain tissue)
        tio.ZNormalization(masking_method=tio.ZNormalization.mean),
        
        # STEP 5: Resizing (Crop or Pad to standardized target volume dimensions)
        tio.CropOrPad(target_shape=target_shape),
        
        # STEP 6: Gaussian Filtering (Smooth high-frequency noise)
        tio.RandomBlur(std=(0.5, 0.5), p=1.0)  # Fixed std=0.5 for deterministic smoothing
    ])
    
    # Apply pipeline to the subject
    processed_subject = transform_pipeline(subject)
    
    # -------------------------------------------------------------------------
    # Formatting into 5D Volumetric Tensor: (B, C, D, H, W)
    # -------------------------------------------------------------------------
    tensor_4d = processed_subject.mri.data  # (1, Depth, Height, Width)
    tensor_5d = tensor_4d.unsqueeze(0)       # (1, 1, Depth, Height, Width)
    
    return tensor_5d


# =============================================================================
# Execution Loop Across Converted NIfTI Files
# =============================================================================
def run_pipeline(input_dir: str, output_dir: str):
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Find all converted .nii.gz files in the input directory
    nifti_files = list(input_path.rglob("*.nii.gz"))
    print(f"Found {len(nifti_files)} NIfTI volume(s) to process.\n")
    
    for idx, file_path in enumerate(nifti_files):
        print(f"[{idx + 1}/{len(nifti_files)}] Processing: {file_path.name}")
        
        try:
            # Run 6-step pipeline
            tensor_5d = preprocess_single_volume(
                nifti_path=file_path, 
                target_shape=(DEPTH, HEIGHT, WIDTH)
            )
            
            # Save preprocessed PyTorch tensor (.pt file)
            out_filename = output_path / f"{file_path.stem.replace('.nii', '')}_preprocessed.pt"
            torch.save(tensor_5d, out_filename)
            
            print(f"   └── Saved 5D Tensor Shape: {tensor_5d.shape} -> {out_filename.name}")
            
        except Exception as e:
            print(f"   └── Error processing {file_path.name}: {e}")

    print("\nPipeline execution complete!")


if __name__ == "__main__":
    run_pipeline(NIFTI_INPUT_DIR, PROCESSED_OUTPUT_DIR)